In [33]:
import json, os
from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.function_tool import FunctionTool
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.tool_context import ToolContext


In [34]:
#import google API keys
credentials = {}

try:
    with open('credentials.json') as file:
        credentials = json.load(file)
        os.environ["GOOGLE_API_KEY"] = credentials['GOOGLE_API_KEY']
        print("API ready to be used")
except FileNotFoundError:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")



API ready to be used


In [35]:
# configure retry options for transient errors
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

In [36]:
# MCP integration with Everything Server
mcp_image_server = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",  # Run MCP server via npx
            args=[
                "-y",  # Argument for npx to auto-confirm install
                "@modelcontextprotocol/server-everything",
            ],
            tool_filter=["getTinyImage"],
        ),
        timeout=30,
    )
)

print("✅ MCP Tool created")

✅ MCP Tool created


In [37]:
# Create image agent with MCP integration
image_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="image_agent",
    instruction="Use the MCP Tool to generate images for user queries",
    tools=[mcp_image_server],
)

In [38]:
runner = InMemoryRunner(agent=image_agent)

In [39]:
response = await runner.run_debug("Provide a sample of tiny square image", verbose=True)

Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno



 ### Created new session: debug_session_id

User > Provide a sample of tiny square image


Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno


image_agent > ```json
[
  {"prompt": "a tiny square image of a single red pixel"}
]
```


In [42]:
from IPython.display import display, Image as IPImage
import base64

for event in response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if hasattr(part, "function_response") and part.function_response:
                for item in part.function_response.response.get("content", []):
                    if item.get("type") == "image":
                        display(IPImage(data=base64.b64decode(item["data"])))